In [5]:
using Turing, Distributions, RData, RCall, CSV, DataFrames, LinearAlgebra
# Function to filter out the specified age groups
function exclude_age_groups(df, ages_to_exclude)
    filter(row -> !(row[:age] in ages_to_exclude), df)
end

function replace_age_groups(dfs, age_map)
    for df in dfs
        df[!, :age] = [get(age_map, age, "0") for age in df[:, :age]]
    end
    return dfs
end

# Function to replace year with index
function replace_year_with_index(df)
    df[!, :year] = [year_to_index[year] for year in df[:, :year]]
    return df
end

# Function to unstack a DataFrame
function unstack_dataframe(df)
    # You might need to adjust the identifier columns as per your DataFrame structure
    identifier_cols = [ :education, :age]
    return unstack(df, identifier_cols, :year, :fertility)
end

unstack_dataframe (generic function with 1 method)

In [6]:
#Import rates
path1 = "/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fxjuliav1.RData"
rates1 = load(path1)

Dict{String, Any} with 1 entry:
  "fxjuliav1" => DictoVec{DataFrame}("No Education"=>168×6 DataFrame…

In [7]:
#Import rates
#mxjuliav1 and mxjuliav2 are created here
#/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year
#/Chapter Mortality/ModellingM/Code/2. BayesianLC/0. Main File BLC.R
path1 = "/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fxjuliav1.RData"
rates1 = load(path1)

# Assigning 'mxjuliav' from 'rates'
fxjuliav1 = rates1["fxjuliav1"]
# Create an ordered list of keys as per your specified order
ordered_keys = [
    "No Education",
    "Primary",
    "Secondary",
    "Post Secondary"
]
# Ordered dataframe using the dictionary
fxjuliav1 = [fxjuliav1[key] for key in ordered_keys]

# Age groups to exclude
ages_to_exclude = ["15--19", "20--24", "55--59", "60--64", "65--69", "70--74", "75+"]
# Apply the filtering to each element in mxjuliav1
fxjuliav1 = [exclude_age_groups(df, ages_to_exclude) for df in fxjuliav1]

# Define the age mapping
age_map = Dict(
    "25--29" => "1",
    "30--34" => "2",
    "35--39" => "3",
    "40--44" => "4",
    "45--49" => "5",
    "50--54" => "6"
)
# Assuming mxjuliav1 is a list (or vector) of DataFrames
fxjuliav1 = replace_age_groups(fxjuliav1, age_map)

# Create a mapping from year to index
year_to_index = Dict(1998:2018 .=> 1:21)
# Apply the function to each DataFrame in your vector
fxjuliav1 = [replace_year_with_index(df) for df in fxjuliav1]


# Iterate through each DataFrame in the vector and drop 'mortality' and 'mx' columns
for df in fxjuliav1
    select!(df, Not([:pop, :fx]))
end

    
# Unstack each DataFrame in mxjuliav1
fxjuliav1_wide = [unstack_dataframe(df) for df in fxjuliav1]
    
    # Iterate through each DataFrame in the vector and drop 'mortality' and 'mx' columns
for df in fxjuliav1_wide
    select!(df, Not([ :education, :age]))
end
        

In [8]:
fxjuliav1_wide

4-element Vector{DataFrame}:
 6×21 DataFrame
 Row │ 1         2         3         4         5         6         7         8 ⋯
     │ Float64?  Float64?  Float64?  Float64?  Float64?  Float64?  Float64?  F ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │  29648.0   30199.0   29117.0   26290.0   24824.0   24211.0   24113.0    ⋯
   2 │  20694.0   21697.0   21673.0   20581.0   19165.0   18763.0   18656.0
   3 │  13516.0   14182.0   14054.0   13154.0   12845.0   12363.0   12315.0
   4 │   4825.0    4872.0    5190.0    4922.0    4621.0    4692.0    4801.0
   5 │    609.0     554.0     563.0     527.0     495.0     469.0     486.0    ⋯
   6 │     41.0      42.0      48.0      37.0      26.0      37.0      50.0
                                                              14 columns omitted
 6×21 DataFrame
 Row │ 1         2         3         4         5         6         7         8 ⋯
     │ Float64?  Float64?  Float64?  Float64?  Float64?  Float64?  F

In [9]:
fxjuliav1

4-element Vector{DataFrame}:
 126×4 DataFrame
 Row │ year   age     education     fertility 
     │ Int64  String  String        Float64   
─────┼────────────────────────────────────────
   1 │     1  1       No Education    29648.0
   2 │     1  2       No Education    20694.0
   3 │     1  3       No Education    13516.0
   4 │     1  4       No Education     4825.0
   5 │     1  5       No Education      609.0
   6 │     1  6       No Education       41.0
   7 │     2  1       No Education    30199.0
   8 │     2  2       No Education    21697.0
   9 │     2  3       No Education    14182.0
  10 │     2  4       No Education     4872.0
  11 │     2  5       No Education      554.0
  ⋮  │   ⋮      ⋮          ⋮            ⋮
 117 │    20  3       No Education     1421.0
 118 │    20  4       No Education      558.0
 119 │    20  5       No Education       93.0
 120 │    20  6       No Education        4.0
 121 │    21  1       No Education     2006.0
 122 │    21  2       No Education 

In [11]:
# Assuming mxjuliav1 is a vector of DataFrames
# Assuming mxjuliav1 is a vector of DataFrames
fx1 =  fxjuliav1_wide[1]
fx2 =  fxjuliav1_wide[2]
fx3 =  fxjuliav1_wide[3]
fx4 =  fxjuliav1_wide[4]

Row,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,13271.0,14062.0,14245.0,14260.0,14600.0,15569.0,17025.0,18109.0,19238.0,20128.0,38265.0,38276.0,37245.0,37088.0,38090.0,39075.0,42718.0,47021.0,47648.0,50107.0,51267.0
2,14646.0,15084.0,15489.0,14844.0,14907.0,15522.0,16171.0,16812.0,17482.0,18243.0,27968.0,29906.0,30622.0,32330.0,33670.0,35448.0,38628.0,41253.0,40372.0,41450.0,41195.0
3,7021.0,7699.0,8236.0,8162.0,8361.0,8791.0,9241.0,9702.0,9682.0,9902.0,13571.0,13484.0,13658.0,14382.0,14852.0,15974.0,18151.0,20465.0,21034.0,22303.0,22775.0
4,1228.0,1376.0,1512.0,1570.0,1608.0,1749.0,1982.0,2075.0,2224.0,2212.0,2919.0,3045.0,3024.0,3046.0,3132.0,3139.0,3325.0,3670.0,3836.0,4264.0,4482.0
5,49.0,58.0,77.0,71.0,68.0,107.0,86.0,92.0,115.0,144.0,170.0,172.0,180.0,165.0,185.0,204.0,230.0,212.0,245.0,269.0,252.0
6,17.0,13.0,15.0,5.0,9.0,5.0,20.0,13.0,19.0,17.0,44.0,29.0,35.0,39.0,36.0,44.0,29.0,56.0,39.0,34.0,46.0


In [12]:
CSV.write("/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fx1.csv", fx1)
CSV.write("/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fx2.csv", fx2)
CSV.write("/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fx3.csv", fx3)
CSV.write("/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fx4.csv", fx4)

"/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/Data_Created/fx4.csv"